In [1]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *
import util_moments as yum

yu.setpath('plot_paper')

enss=['b','c','d','e']

# isovectors

In [2]:
folder='step1'
key2phy_A20_pre=yu.load_pkl_reg(f'{folder}/key2phy_A20_stout',pathlabel='analysis_xJ_syst')
key2phy_B20_pre=yu.load_pkl_reg(f'{folder}/key2phy_B20_stout',pathlabel='analysis_xJ_syst')
stouts_jointlinear=[7,10,13,15,20]

In [3]:
# together

stouts=stouts_jointlinear
fig,axs=yu.getFigAxs(2,1,Lrow=3,Lcol=8,sharex=True)

fontsize=24

ax=axs[0,0]
yu.addRefLine(ax,0,hv='v')
key2phy=key2phy_A20_pre
ax.tick_params(axis="both", labelsize=fontsize)
for ij,j in enumerate(['jv1','jv2','jv3']):
    vflas=['u-d','u+d-2s','u+d+s-3c'][ij]
    color=yu.colors8[ij]; fmt=yu.fmts8[ij]
    for iens,ens in enumerate(enss):
        t=key2phy[(ens,j)]
        mean,err=yu.jackme(t)
        
        plt_x=yu.ens2a[ens]**2+ij*0.0001/2; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt,label=rf'$\langle x \rangle_{{{vflas}}}$' if iens==0 else None, markersize=13, capsize=13)

    t=key2phy[('a=#_MA',j)]
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    ax.fill_between(x, ymin, ymax, color=color, alpha=0.2)
    

ax.set_ylabel(r'$\langle x \rangle_{q^{ns}}$',size=fontsize)
ax.set_xlim([-0.001,0.0075])
ax.set_ylim([0.1,0.9])
ax.legend(loc="upper right", frameon=True, ncol=5,
                  fontsize=fontsize-6, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

ax=axs[1,0]
yu.addRefLine(ax,0)
yu.addRefLine(ax,0,hv='v')
key2phy=key2phy_B20_pre
ax.tick_params(axis="both", labelsize=fontsize)
for ij,j in enumerate(['jv1','jv2','jv3']):
    vflas=['u-d','u+d-2s','u+d+s-3c'][ij]
    color=yu.colors8[ij]; fmt=yu.fmts8[ij]
    for iens,ens in enumerate(enss):
        t=key2phy[(ens,j)]
        mean,err=yu.jackme(t)
        
        plt_x=yu.ens2a[ens]**2+ij*0.0001/2; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt,label=rf'$B_{{20}}^{{{vflas}}}$' if iens==0 else None, markersize=13, capsize=13)

    t=key2phy[('a=#_MA',j)]
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    ax.fill_between(x, ymin, ymax, color=color, alpha=0.2)

ax.tick_params(axis="both", labelsize=fontsize)

ax.set_xlim([-0.001,0.0075])
# ax.legend(loc="upper right", frameon=True, ncol=5,
#                   fontsize=fontsize-12, handlelength=0.9, columnspacing=0.45,
#                   handletextpad=0.25, borderpad=0.25)

ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set_ylabel(r'$B_{20}^{q^{ns}}$',size=fontsize)

ax.set_yticks(np.arange(-0.1,0.5,0.1))
ax.set_ylim([-0.18,0.48])
ax.legend(loc="upper right", frameon=True, ncol=5,
                  fontsize=fontsize-6, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
ax.xaxis.get_offset_text().set_fontsize(fontsize)
yu.finalizePlot(f'ce_AB20_v123')

# gluon

In [4]:
# together

stouts=stouts_jointlinear
fig,axs=yu.getFigAxs(2,1,Lrow=3,Lcol=8,sharex=True)

key2phy_A20=key2phy_A20_pre.copy()

y_jk=yu.superjackknife([np.transpose([key2phy_A20[(ens,f'jg;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
fit=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
pars_jk,chi2_jk,Ndof,Nwarning=fit
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

# def fitfunc(pars):
#     return [pars[0] for ens in enss for ist,stout in enumerate(stouts)]
# fit2=yu.jackfit(fitfunc,y_jk,pars0=[0.4])
# fits=[fit,fit2]
# pars_jk,probs_jk=yu.jackMA(fits)

fontsize=24

ax=axs[0,0]
ax.tick_params(axis="both", labelsize=fontsize)
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    # if nst==10:
        # phy_A20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    # ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_A20[(ens,f'jg;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt,label=nst if iens==0 else None, markersize=13, capsize=13, mfc='white' if nst==10 else None)
        
    t=key2phy_A20[('a=#_MA',f'jg;stout{nst}')][:,0]
    # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
    # print(yu.jackme_un2str(t))
    mean,err=yu.jackme(t)
    plt_x=-0.001/10*(ist+1); plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt, markersize=13, capsize=13,  mfc='white' if nst==10 else None)
    
    if nst in [10]:
        mean,err=yu.jackme(key2phy_A20[('a=#_MA',f'jg;stout{nst}')])
        x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
        ax.fill_between(x, ymin, ymax, color=color, alpha=0.2)
        print(yu.un2str(mean[0],err[0]))

ax.set_xlim([-0.001,0.0075])
# ax.set_ylim([0.15,0.65])
ax.legend(loc="upper right", frameon=True, ncol=5,
                  fontsize=fontsize-6, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

yu.addRefLine(ax,0,'v')
# ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
# ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set_ylabel(r'$\langle x \rangle_g$',size=fontsize)
ax.set_ylim([0.15,0.65])
ax.set_yticks([0.2,0.3,0.4,0.5,0.6])

# ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
# ax.xaxis.get_offset_text().set_fontsize(fontsize)
# yu.finalizePlot('jointLinear_avgx_quark_single')


key2phy_B20=key2phy_B20_pre.copy()

y_jk=yu.superjackknife([np.transpose([key2phy_B20[(ens,f'jg;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

ax=axs[1,0]
ax.tick_params(axis="both", labelsize=fontsize)
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    # if nst==10:
    #     phy_B20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    # ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_B20[(ens,f'jg;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,label=nst if iens==0 else None, markersize=13, capsize=13, mfc='white' if nst==10 else None)
        
    t=key2phy_B20[('a=#_MA',f'jg;stout{nst}')][:,0]
    # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
    # print(yu.jackme_un2str(t))
    mean,err=yu.jackme(t)
    plt_x=-0.001/10*(ist+1); plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt, markersize=13, capsize=13, mfc='white' if nst==10 else None)
    
    if nst in [10]:
        mean,err=yu.jackme(key2phy_B20[('a=#_MA',f'jg;stout{nst}')])
        x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
        ax.fill_between(x, ymin, ymax, color=color, alpha=0.2)
        print(yu.un2str(mean[0],err[0]))

ax.set_xlim([-0.001,0.0075])
# ax.legend(loc="upper right", frameon=True, ncol=5,
#                   fontsize=fontsize-12, handlelength=0.9, columnspacing=0.45,
#                   handletextpad=0.25, borderpad=0.25)

ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set_ylabel(r'$B_{20}^{g}$',size=fontsize)
ax.set_ylim([-0.18,0.18])
# ax.set_yticks([-0.2,-0.1,0,0.1,0.2])

yu.addRefLine(ax,0,'v')
yu.addRefLine(ax,0)

ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
ax.xaxis.get_offset_text().set_fontsize(fontsize)
yu.finalizePlot(f'jointLinear_AB20_gluon')

0.388(51) 0.8953687391680715 14
0.372(30)
0.032(51) 1.947095120770475 14
0.047(44)


In [5]:
stouts=stouts_jointlinear

key2phy_A20={(ens,f'jg;stout{nst}'):key2phy_A20_pre[(ens,f'jg;stout{nst}')] for ens in enss for nst in stouts }
for nst in stouts:
    key2phy_A20[('a=#_const',f'jg;stout{nst}')]=key2phy_A20_pre[('a=#_const',f'jg;stout{nst}')]
    key2phy_A20[('a=#_linear',f'jg;stout{nst}')]=key2phy_A20_pre[('a=#_linear',f'jg;stout{nst}')]
    key2phy_A20[('a=#_MA',f'jg;stout{nst}')]=key2phy_A20_pre[('a=#_MA',f'jg;stout{nst}')]

y_jk=yu.superjackknife([np.transpose([key2phy_A20[(ens,f'jg;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

fontsize=36

fig,axs=yu.getFigAxs(1,2,Lrow=7,Lcol=7,sharey=True)
ax=axs[0,0]
ax.tick_params(axis="both", labelsize=fontsize)
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    if nst==10:
        phy_A20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_A20[(ens,f'jg;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt,label=nst if iens==0 else None, markersize=13, capsize=13, mfc='white' if nst==10 else None)
    
    if nst==10:
        t=key2phy_A20[('a=#_MA',f'jg;stout{nst}')][:,0]
        # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
        # print(yu.jackme_un2str(t))
        mean,err=yu.jackme(t)
        plt_x=-0.001/10*(ist+1)*1.5; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt, markersize=13, capsize=13,  mfc='white' if nst==10 else None)
        
# for nst in [10]:
#     mean,err=yu.jackme(key2phy_A20[('a=#_linear',f'jg;stout{nst}')])
#     x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
#     ax.fill_between(x, ymin, ymax, color='black', alpha=0.1)

ax.set_xlim([-0.001,0.0075])
ax.set_ylim([0.15,0.65])
ax.legend(loc="upper right", frameon=True, ncol=5,
                  fontsize=fontsize-12, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

yu.addRefLine(ax,0,'v')
ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007], xticklabels=[0,1,2,3,4,5,6,7])
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set_ylabel(r'$\langle x \rangle_g$',size=fontsize)
ax.set_yticks([0.2,0.3,0.4,0.5,0.6])

ax=axs[0,1]
ax.tick_params(axis="both", labelsize=fontsize)
ens2phy={}
for iens,ens in enumerate(enss):
    ens2phy[ens]=np.transpose([key2phy_A20_pre[(ens,f'jg;stout{nst}')] for nst in stouts])
    # print(ens,[yu.jackme_un2str(key2phy_A20_pre[(ens,f'jg;stout{nst}')]) for nst in stouts])
    pars_jk,chi2_jk,Ndof=yu.doFit_const(ens2phy[ens],corrQ=False)
    # print(yu.jackme_un2str(pars_jk[:,0]))
    ens2phy[ens]=pars_jk[:,0]
    mean,err=yu.jackme(pars_jk)
    plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr, color='black', markersize=13, capsize=13)
fits=yu.doFits_continuumExtrapolation(ens2phy,lat_a2s_plt=yum.lat_a2s_plt,fitlabels=['linear'])
pars_jk,probs_jk=yu.jackMA(fits)
print(yu.jackme_un2str(pars_jk[:,0]))
mean,err=yu.jackme(pars_jk)
x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
ax.fill_between(x, ymin, ymax, color='black', alpha=0.1)
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
ax.xaxis.get_offset_text().set_fontsize(fontsize)
yu.addRefLine(ax,0,'v')
yu.finalizePlot('jointLinear_avgx',closeQ=True)

0.388(51) 0.8953687391680715 14
0.375(56)


In [6]:
stouts=stouts_jointlinear

key2phy_B20={(ens,f'jg;stout{nst}'):key2phy_B20_pre[(ens,f'jg;stout{nst}')] for ens in enss for nst in stouts }
for nst in stouts:
    key2phy_B20[('a=#_const',f'jg;stout{nst}')]=key2phy_B20_pre[('a=#_const',f'jg;stout{nst}')]
    key2phy_B20[('a=#_linear',f'jg;stout{nst}')]=key2phy_B20_pre[('a=#_linear',f'jg;stout{nst}')]
    key2phy_B20[('a=#_MA',f'jg;stout{nst}')]=key2phy_B20_pre[('a=#_MA',f'jg;stout{nst}')]

y_jk=yu.superjackknife([np.transpose([key2phy_B20[(ens,f'jg;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

fig,axs=yu.getFigAxs(1,2,Lrow=7,Lcol=7,sharey=True)
ax=axs[0,0]
ax.tick_params(axis="both", labelsize=fontsize)
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    if nst==10:
        phy_B20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_B20[(ens,f'jg;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt,label=nst if iens==0 else None, markersize=13, capsize=13,  mfc='white' if nst==10 else None)
        
    if nst==10:
        t=key2phy_B20[('a=#_MA',f'jg;stout{nst}')][:,0]
        # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
        # print(yu.jackme_un2str(t))
        mean,err=yu.jackme(t)
        plt_x=-0.001/10*(ist+1)*1.5; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt, markersize=13, capsize=13,  mfc='white' if nst==10 else None)

ax.set_xlim([-0.001,0.0075])
ax.legend(loc="upper right", frameon=True, ncol=5,
                  fontsize=fontsize-12, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007], xticklabels=[0,1,2,3,4,5,6,7])
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set_ylabel(r'$B_{20}^{g}$',size=fontsize)
ax.set_ylim([-0.18,0.28])
ax.set_yticks([-0.1,0,0.1,0.2])

yu.addRefLine(ax,0,'v')
yu.addRefLine(ax,0)

ax=axs[0,1]
ax.tick_params(axis="both", labelsize=fontsize)
ens2phy={}
for iens,ens in enumerate(enss):
    ens2phy[ens]=np.transpose([key2phy_B20_pre[(ens,f'jg;stout{nst}')] for nst in stouts])
    pars_jk,chi2_jk,Ndof=yu.doFit_const(ens2phy[ens],corrQ=False)
    ens2phy[ens]=pars_jk[:,0]
    mean,err=yu.jackme(pars_jk)
    plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr, color='black', markersize=13, capsize=13)
fits=yu.doFits_continuumExtrapolation(ens2phy,lat_a2s_plt=yum.lat_a2s_plt,fitlabels=['linear'])
pars_jk,probs_jk=yu.jackMA(fits)
print(yu.jackme_un2str(pars_jk[:,0]))
mean,err=yu.jackme(pars_jk)
x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
ax.fill_between(x, ymin, ymax, color='black', alpha=0.1)

ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
ax.xaxis.get_offset_text().set_fontsize(fontsize)
yu.addRefLine(ax,0,'v')
yu.addRefLine(ax,0)
yu.finalizePlot(f'jointLinear_B20')

0.032(51) 1.947095120770475 14
0.073(58)


# quark

In [7]:
# together

stouts=stouts_jointlinear
fig,axs=yu.getFigAxs(2,1,Lrow=3,Lcol=8,sharex=True)

key2phy_A20=key2phy_A20_pre.copy()

y_jk=yu.superjackknife([np.transpose([key2phy_A20[(ens,f'jq;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
fit=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
pars_jk,chi2_jk,Ndof,Nwarning=fit
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

# def fitfunc(pars):
#     return [pars[0] for ens in enss for ist,stout in enumerate(stouts)]
# fit2=yu.jackfit(fitfunc,y_jk,pars0=[0.4])
# fits=[fit,fit2]
# pars_jk,probs_jk=yu.jackMA(fits)

fontsize=24

ax=axs[0,0]
ax.tick_params(axis="both", labelsize=fontsize)
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    # if nst==10:
        # phy_A20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    # ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_A20[(ens,f'jq;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt,label=nst if iens==0 else None, markersize=13, capsize=13, mfc='white' if nst==10 else None)
        
    t=key2phy_A20[('a=#_MA',f'jq;stout{nst}')][:,0]
    # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
    # print(yu.jackme_un2str(t))
    mean,err=yu.jackme(t)
    plt_x=-0.001/10*(ist+1); plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt, markersize=13, capsize=13,  mfc='white' if nst==10 else None)
    
    if nst in [10]:
        mean,err=yu.jackme(key2phy_A20[('a=#_MA',f'jq;stout{nst}')])
        x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
        ax.fill_between(x, ymin, ymax, color=color, alpha=0.2)

ax.set_xlim([-0.001,0.0075])
# ax.set_ylim([0.15,0.65])
ax.legend(loc="upper right", frameon=True, ncol=5,
                  fontsize=fontsize-6, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

yu.addRefLine(ax,0,'v')
# ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
# ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set_ylabel(r'$\langle x \rangle_{q^s}$',size=fontsize)
ax.set_ylim([0.45,0.85])
ax.set_yticks([0.55,0.65,0.75])

# ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
# ax.xaxis.get_offset_text().set_fontsize(fontsize)
# yu.finalizePlot('jointLinear_avgx_quark_single')


key2phy_B20=key2phy_B20_pre.copy()

y_jk=yu.superjackknife([np.transpose([key2phy_B20[(ens,f'jq;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

ax=axs[1,0]
ax.tick_params(axis="both", labelsize=fontsize)
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    # if nst==10:
    #     phy_B20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    # ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_B20[(ens,f'jq;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,label=nst if iens==0 else None, markersize=13, capsize=13, mfc='white' if nst==10 else None)
        
    t=key2phy_B20[('a=#_MA',f'jq;stout{nst}')][:,0]
    # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
    # print(yu.jackme_un2str(t))
    mean,err=yu.jackme(t)
    plt_x=-0.001/10*(ist+1); plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt, markersize=13, capsize=13, mfc='white' if nst==10 else None)
    
    if nst in [10]:
        mean,err=yu.jackme(key2phy_B20[('a=#_MA',f'jq;stout{nst}')])
        x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
        ax.fill_between(x, ymin, ymax, color=color, alpha=0.2)

ax.set_xlim([-0.001,0.0075])
# ax.legend(loc="upper right", frameon=True, ncol=5,
#                   fontsize=fontsize-12, handlelength=0.9, columnspacing=0.45,
#                   handletextpad=0.25, borderpad=0.25)

ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set_ylabel(r'$B_{20}^{q^s}$',size=fontsize)
ax.set_ylim([-0.18,0.18])
# ax.set_yticks([-0.2,-0.1,0,0.1,0.2])

yu.addRefLine(ax,0,'v')
yu.addRefLine(ax,0)

ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
ax.xaxis.get_offset_text().set_fontsize(fontsize)
yu.finalizePlot(f'jointLinear_AB20_quark')

0.69(11) 0.6385876249018122 14
0.017(96) 1.860312910584334 14


In [8]:
stouts=stouts_jointlinear

key2phy_A20=key2phy_A20_pre.copy()

y_jk=yu.superjackknife([np.transpose([key2phy_A20[(ens,f'jq;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
fit=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
pars_jk,chi2_jk,Ndof,Nwarning=fit
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

# def fitfunc(pars):
#     return [pars[0] for ens in enss for ist,stout in enumerate(stouts)]
# fit2=yu.jackfit(fitfunc,y_jk,pars0=[0.4])
# fits=[fit,fit2]
# pars_jk,probs_jk=yu.jackMA(fits)

fontsize=36

fig,axs=yu.getFigAxs(1,1,Lrow=5,Lcol=8,sharey=True)
ax=axs[0,0]
ax.tick_params(axis="both", labelsize=fontsize)
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    # if nst==10:
        # phy_A20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    # ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_A20[(ens,f'jq;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt,label=nst if iens==0 else None, markersize=13, capsize=13, mfc='white' if nst==10 else None)
        
    t=key2phy_A20[('a=#_MA',f'jq;stout{nst}')][:,0]
    # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
    # print(yu.jackme_un2str(t))
    mean,err=yu.jackme(t)
    plt_x=-0.001/10*(ist+1); plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt, markersize=13, capsize=13,  mfc='white' if nst==10 else None)
    
    if nst in [10]:
        mean,err=yu.jackme(key2phy_A20[('a=#_MA',f'jq;stout{nst}')])
        x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
        ax.fill_between(x, ymin, ymax, color=color, alpha=0.2)

ax.set_xlim([-0.001,0.0075])
# ax.set_ylim([0.15,0.65])
ax.legend(loc="upper right", frameon=True, ncol=5,
                  fontsize=fontsize-12, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

yu.addRefLine(ax,0,'v')
ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set_ylabel(r'$\langle x \rangle_{q^s}$',size=fontsize)
ax.set_yticks([0.4,0.5,0.6,0.7,0.8,0.9])

ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
ax.xaxis.get_offset_text().set_fontsize(fontsize)
yu.finalizePlot('jointLinear_avgx_quark_single')

0.69(11) 0.6385876249018122 14


In [9]:
stouts=stouts_jointlinear

key2phy_B20=key2phy_B20_pre.copy()

y_jk=yu.superjackknife([np.transpose([key2phy_B20[(ens,f'jq;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

fig,axs=yu.getFigAxs(1,1,Lrow=5,Lcol=8,sharey=True)
ax=axs[0,0]
ax.tick_params(axis="both", labelsize=fontsize)
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    # if nst==10:
    #     phy_B20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    # ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_B20[(ens,f'jq;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,label=nst if iens==0 else None, markersize=13, capsize=13, mfc='white' if nst==10 else None)
        
    t=key2phy_B20[('a=#_MA',f'jq;stout{nst}')][:,0]
    # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
    # print(yu.jackme_un2str(t))
    mean,err=yu.jackme(t)
    plt_x=-0.001/10*(ist+1); plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt, markersize=13, capsize=13, mfc='white' if nst==10 else None)
    
    if nst in [10]:
        mean,err=yu.jackme(key2phy_B20[('a=#_MA',f'jq;stout{nst}')])
        x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
        ax.fill_between(x, ymin, ymax, color=color, alpha=0.2)

ax.set_xlim([-0.001,0.0075])
ax.legend(loc="upper right", frameon=True, ncol=5,
                  fontsize=fontsize-12, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set_ylabel(r'$B_{20}^{q^s}$',size=fontsize)
# ax.set_ylim([-0.18,0.28])
ax.set_yticks([-0.2,-0.1,0,0.1,0.2])

yu.addRefLine(ax,0,'v')
yu.addRefLine(ax,0)

ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
ax.xaxis.get_offset_text().set_fontsize(fontsize)
yu.finalizePlot(f'jointLinear_B20_quark_single')

0.017(96) 1.860312910584334 14


In [10]:
stouts=stouts_jointlinear

key2phy_A20=key2phy_A20_pre.copy()

y_jk=yu.superjackknife([np.transpose([key2phy_A20[(ens,f'jq;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
fit=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
pars_jk,chi2_jk,Ndof,Nwarning=fit
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

# def fitfunc(pars):
#     return [pars[0] for ens in enss for ist,stout in enumerate(stouts)]
# fit2=yu.jackfit(fitfunc,y_jk,pars0=[0.4])
# fits=[fit,fit2]
# pars_jk,probs_jk=yu.jackMA(fits)

fontsize=32

fig,axs=yu.getFigAxs(1,2,Lrow=5.6,Lcol=7.4,sharey=True)
ax=axs[0,0]
ax.tick_params(axis="both", labelsize=fontsize)
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    # if nst==10:
        # phy_A20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    # ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_A20[(ens,f'jq;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt,label=nst if iens==0 else None, markersize=13, capsize=13)
        
    t=key2phy_A20[('a=#_MA',f'jq;stout{nst}')][:,0]
    # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
    # print(yu.jackme_un2str(t))
    mean,err=yu.jackme(t)
    plt_x=-0.001/10*(ist+1); plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt, markersize=13, capsize=13)
    
# for nst in [10]:
#     mean,err=yu.jackme(key2phy_A20[('a=#_linear',f'jg;stout{nst}')])
#     x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
#     ax.fill_between(x, ymin, ymax, color='black', alpha=0.1)

ax.set_xlim([-0.001,0.0075])
# ax.set_ylim([0.15,0.65])
ax.legend(loc="upper right", frameon=True, ncol=5,
                  fontsize=fontsize-12, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

yu.addRefLine(ax,0,'v')
ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007], xticklabels=[0,1,2,3,4,5,6,7])
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set_ylabel(r'$\langle x \rangle_{q^s}$',size=fontsize)
ax.set_yticks([0.2,0.3,0.4,0.5,0.6])

ax=axs[0,1]
ax.tick_params(axis="both", labelsize=fontsize)
ens2phy={}
for iens,ens in enumerate(enss):
    ens2phy[ens]=np.transpose([key2phy_A20_pre[(ens,f'jq;stout{nst}')] for nst in stouts])
    # print(ens,[yu.jackme_un2str(key2phy_A20_pre[(ens,f'jg;stout{nst}')]) for nst in stouts])
    pars_jk,chi2_jk,Ndof=yu.doFit_const(ens2phy[ens],corrQ=False)
    # print(yu.jackme_un2str(pars_jk[:,0]))
    ens2phy[ens]=pars_jk[:,0]
    mean,err=yu.jackme(pars_jk)
    plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr, color='black', markersize=13, capsize=13)
fits=yu.doFits_continuumExtrapolation(ens2phy,lat_a2s_plt=yum.lat_a2s_plt,fitlabels=['const','linear'])
pars_jk,probs_jk=yu.jackMA(fits)
print(yu.jackme_un2str(pars_jk[:,0]))
mean,err=yu.jackme(pars_jk)
x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
ax.fill_between(x, ymin, ymax, color='black', alpha=0.1)
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
ax.xaxis.get_offset_text().set_fontsize(fontsize)
yu.addRefLine(ax,0,'v')
yu.finalizePlot('jointLinear_avgx_quark')

0.69(11) 0.6385876249018122 14
0.620(46)


In [11]:
stouts=stouts_jointlinear

key2phy_B20=key2phy_B20_pre.copy()

y_jk=yu.superjackknife([np.transpose([key2phy_B20[(ens,f'jq;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

fig,axs=yu.getFigAxs(1,2,sharey=True)
ax=axs[0,0]
ax.tick_params(axis="both", labelsize=fontsize)
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    # if nst==10:
    #     phy_B20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_B20[(ens,f'jq;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,label=nst if iens==0 else None, markersize=13, capsize=13)
        
    t=key2phy_B20[('a=#_MA',f'jq;stout{nst}')][:,0]
    # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
    # print(yu.jackme_un2str(t))
    mean,err=yu.jackme(t)
    plt_x=-0.001/10*(ist+1); plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt, markersize=13, capsize=13)

ax.set_xlim([-0.001,0.0075])
ax.legend(loc="upper right", frameon=True, ncol=5,
                  fontsize=fontsize-12, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007], xticklabels=[0,1,2,3,4,5,6,7])
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.set_ylabel(r'$B_{20}^{q^s}$',size=fontsize)
ax.set_ylim([-0.18,0.28])
ax.set_yticks([-0.1,0,0.1,0.2])

yu.addRefLine(ax,0,'v')
yu.addRefLine(ax,0)

ax=axs[0,1]
ax.tick_params(axis="both", labelsize=fontsize)
ens2phy={}
for iens,ens in enumerate(enss):
    ens2phy[ens]=np.transpose([key2phy_B20_pre[(ens,f'jq;stout{nst}')] for nst in stouts])
    pars_jk,chi2_jk,Ndof=yu.doFit_const(ens2phy[ens],corrQ=False)
    ens2phy[ens]=pars_jk[:,0]
    mean,err=yu.jackme(pars_jk)
    plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr, color='black', markersize=13, capsize=13)
fits=yu.doFits_continuumExtrapolation(ens2phy,lat_a2s_plt=yum.lat_a2s_plt,fitlabels=['linear','const'])
pars_jk,probs_jk=yu.jackMA(fits)
print(yu.jackme_un2str(pars_jk[:,0]))
mean,err=yu.jackme(pars_jk)
x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
ax.fill_between(x, ymin, ymax, color='black', alpha=0.1)

ax.set(xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
ax.set_xlabel(r"$a^2\ [\mathrm{fm}^2]$",size=fontsize)
ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
ax.xaxis.get_offset_text().set_fontsize(fontsize)
yu.addRefLine(ax,0,'v')
yu.addRefLine(ax,0)
yu.finalizePlot(f'jointLinear_B20_quark')

0.017(96) 1.860312910584334 14
-0.025(37)


# others

In [2]:
folders=['step1','case3','step2','case6','case7']

key2phy_A20s=[yu.load_pkl_reg(f'{folder}/key2phy_A20',pathlabel='analysis_xJ_syst') for folder in folders]
key2phy_B20s=[yu.load_pkl_reg(f'{folder}/key2phy_B20',pathlabel='analysis_xJ_syst') for folder in folders]
key2phy_Js=[yu.load_pkl_reg(f'{folder}/key2phy_J',pathlabel='analysis_xJ_syst') for folder in folders]

for key2phys in [key2phy_A20s,key2phy_B20s,key2phy_Js]:
    for key2phy in key2phys:
        for ens in enss+['a=#_final']:
            key2phy[(ens,'j+')]=key2phy[(ens,'ju')]+key2phy[(ens,'jd')]

In [3]:
ens2Njk={'b':725,'c':400,'d':493,'e':516}
Nsupjk=sum([ens2Njk[ens] for ens in enss])

path='/p/project1/ngff/li47/code/projectData/05_moments/fromOthers/gA_fromChristos.h5'
key2phy={}
with h5py.File(path) as f:
    for ens in enss:
        for fla in ['u','d','s','c']:
            t=f[fla][ens.upper()][:]
            key2phy[(ens,f'j{fla}')]=t
        for sgn in ['+','-']:
            t=f[f'u{sgn}d_conn'][ens.upper()][:]
            key2phy[(ens,f'j{sgn};conn')]=t
            
for ens in enss:
    key2phy[(ens,'j+')]=key2phy[(ens,'ju')]+key2phy[(ens,'jd')]
    key2phy[(ens,'j-')]=key2phy[(ens,'ju')]-key2phy[(ens,'jd')]
    key2phy[(ens,f'ju;conn')] = (key2phy[(ens,f'j+;conn')]+key2phy[(ens,f'j-;conn')])/2
    key2phy[(ens,f'jd;conn')] = (key2phy[(ens,f'j+;conn')]-key2phy[(ens,f'j-;conn')])/2
                        
    for j in yum.j2j1.keys():
        key2phy[(ens,j)]=np.sum([key2phy[(ens,j1)]*c for c,j1 in yum.j2j1[j]],axis=0)
    key2phy[(ens,f'jq;conn')]=key2phy[(ens,f'j+;conn')]
    key2phy[(ens,f'jv1;conn')]=key2phy[(ens,f'j-;conn')]
        
    # key2phy[(ens,f'jtot')]=key2phy[(ens,f'jq')]
    # key2phy[(ens,f'jtot;conn')]=key2phy[(ens,f'jq;conn')]
    
    del key2phy[(ens,'j+;conn')]
    del key2phy[(ens,'j-;conn')]
    del key2phy[(ens,'j-')]
    
for ens,j in key2phy.keys():
    t=key2phy[(ens,j)]
    mean,err=yu.jackme(t)
    t=yu.jackknife_pseudo(mean,err,n=ens2Njk[ens])[:,0]
    key2phy[(ens,j)]=t

js=list(set([j for ens,j in key2phy.keys()])); js.sort()
for j in js:
    ens2dat={ens:key2phy[(ens,j)] for ens in enss}
    fits=yu.doFits_continuumExtrapolation(ens2dat,lat_a2s_plt=yum.lat_a2s_plt,fitlabels=['const','const-1','const-2','linear','linear-1'],supjackQ=False)
    mean,err,probs=yu.modelAvg(fits)
    
    t=yu.jackknife_pseudo(mean,err,n=Nsupjk)
    key2phy[('a=#_final',j)]=t

key2phy_gAs=[key2phy.copy(),key2phy.copy(),key2phy.copy(),key2phy.copy(),key2phy.copy()]
key2phy_DeltaSigmaBy2s=[{key:key2phy[key]/2 for key in key2phy.keys()} for key2phy in key2phy_gAs]
key2phy_Ls=[{key:key2phy_J[key]-key2phy_DeltaSigmaBy2[key] for key in key2phy_DeltaSigmaBy2.keys()} for key2phy_J,key2phy_DeltaSigmaBy2 in zip(key2phy_Js,key2phy_DeltaSigmaBy2s)]

In [5]:
capsize_global=6; capthick_global=0.5
def get_key2syst(key2phys):
    key2syst={}
    for key in key2phys[0].keys():
        if 'conn' in key[1]:
            continue
        m0=np.mean(key2phys[0][key],axis=0)
        ds=np.array([np.mean(key2phy[key],axis=0)-m0 for key2phy in key2phys])
        t=np.sum(ds[1:]**2,axis=0)
        key2syst[key]=np.sqrt(t)
    return key2syst

def get_key2syst_2(key2phys):
    key2syst={}
    for key in key2phys[0].keys():
        if 'conn' in key[1]:
            continue
        m0=np.mean(key2phys[0][key],axis=0)
        ds=np.array([np.mean(key2phy[key],axis=0)-m0 for key2phy in key2phys])
        t1=np.sum(ds[1:3]**2,axis=0)
        t2=np.sum(ds[3:4]**2,axis=0)
        t3=np.sum(ds[4:5]**2,axis=0)
        
        ens,j=key
        if 'jg' in j or 'jtot' in j:
            key2syst[key]=(np.sqrt(t1),np.sqrt(t2),np.sqrt(t3))
        else:
            key2syst[key]=(np.sqrt(t1),np.sqrt(t2))
    return key2syst


def plot_A20_B20_J(fig,axs,key2phy,which,ylabelQ=True,ce='final',key2syst=None,syst0onlyQ=True,rightmostQ=False):
    sty = {"jtot": ("gray", "o"), "jq": ("purple", "d"), "jg": ("cyan", "s"),
           "ju": ("red", "^"), "jd": ("green", "v"), "js": ("blue", "<"), "jc": ("orange", ">")}

    if which == "A20":
        rows = [(["jtot", "jq", "jg"], r"$\langle x\rangle_{q^s,g}$", (0.20, 1.40), [0.4,0.6,0.8,1.0,1.2], 1.0),
                (["ju", "jd", "js", "jc"], r"$\langle x\rangle_q$", (-0.10, 0.58), [0.0, 0.2, 0.4], 0.0)]
        lab = dict(jtot=r"$\langle x\rangle_N$", jq=r"$\langle x\rangle_{q^s}$", jg=r"$\langle x\rangle_g$",
                   ju=r"$\langle x\rangle_u$", jd=r"$\langle x\rangle_d$",
                   js=r"$\langle x\rangle_s$", jc=r"$\langle x\rangle_c$")
    elif which == "B20":
        rows = [(["jtot"], r"$B_{20}^N$", (-0.5, 0.5), [-0.3, 0.0, 0.3], 0.0),
                (["jq", "jg"], r"$B_{20}^{q^s,g}$", (-0.3, 0.3), [-0.2, 0.0, 0.2], 0.0),
                (["ju", "jd", "js", "jc"], r"$B_{20}^q$", (-0.22, 0.22), [-0.1, 0.0, 0.1], 0.0)]
        lab = {j: rf"$B_{{20}}^{{{s}}}$" for j, s in
               zip(["jtot", "jq", "jg", "ju", "jd", "js", "jc"], ["N", "q^s", "g", "u", "d", "s", "c"])}
    elif which == 'J':
        rows = [(["jtot", "jq", "jg"], r"$J_{q^s,g}$", (0.03, 0.80), [0.1, 0.3, 0.5, 0.7], 0.5),
                (["ju", "jd", "js", "jc"], r"$J_q$", (-0.03, 0.35), [0.0, 0.1, 0.2, 0.3], 0.0)]
        lab = {j: rf"$J_{{{s}}}$" for j, s in
               zip(["jtot", "jq", "jg", "ju", "jd", "js", "jc"], ["N", "q^s", "g", "u", "d", "s", "c"])}
    elif which == 'DeltaSigmaBy2':
        rows = [(["jq","ju", "jd", "js", "jc"], r"$\frac{1}{2}\Delta\Sigma_q$", (-0.3, 0.7), [-0.2, 0, 0.2, 0.4, 0.6], 0.0)]
        lab = {j: rf"$\frac{{1}}{{2}}\Delta\Sigma_{{{s}}}$" for j, s in
               zip(["jtot", "jq", "jg", "ju", "jd", "js", "jc"], ["N", "N", "g", "u", "d", "s", "c"])}
    elif which == 'L':
        rows = [(["jq","ju", "jd", "js", "jc"], r"$L_q$", (-0.25, 0.45), [-0.2, -0.1, 0, 0.1, 0.2, 0.3, 0.4], 0.0)]
        lab = {j: rf"$L_{{{s}}}$" for j, s in
               zip(["jtot", "jq", "jg", "ju", "jd", "js", "jc"], ["N", "N", "g", "u", "d", "s", "c"])}
    else:
        1/0

    for ax, (js, ylabel, ylim, yticks, ref) in zip(axs, rows):
        for ij,j in enumerate(js):
            c, m = sty[j]
            x = np.asarray(yum.lat_a2s_plt)
            y, e = map(np.asarray, yu.jackme(key2phy[("a=#_" + ce, j)]))

            ax.plot(x, y, "--", color=c, lw=2)
            ax.fill_between(x, y - e, y + e, color=c, alpha=0.2)

            xs, ys, es = zip(*[(yu.ens2a[ens]**2 + 0.001/10*ij, *yu.jackme(key2phy[(ens, j)])) for iens,ens in enumerate(enss)])
            ax.errorbar(xs, ys, yerr=es, fmt=m, color=c, label=lab[j], capsize=capsize_global, capthick=capthick_global, lw=2, ms=6)
            
            if key2syst is not None:
                x = np.asarray(yum.lat_a2s_plt)
                y, e = map(np.asarray, yu.jackme(key2phy[("a=#_" + ce, j)]))
                if syst0onlyQ:
                    markersize= 16 if which not in ['DeltaSigmaBy2'] else 16
                    ax.errorbar(0.001/5,y[0],e[0], color=c, fmt='*', markersize=markersize,capsize=capsize_global, capthick=capthick_global, mfc='white')
                    # print(which,j,y[0],e[0])
                    e = np.sqrt(e**2 + key2syst[("a=#_" + ce, j)]**2)
                    # print(e[0])
                    ax.errorbar(0.001/5,y[0],e[0], color=c, fmt='*', markersize=markersize,capsize=capsize_global, capthick=capthick_global, mfc='white')
                else:
                    e = np.sqrt(e**2 + key2syst[("a=#_" + ce, j)]**2)
                    ax.plot(x, y, "--", color=c, lw=2)
                    ax.fill_between(x, y - e, y + e, color=c, alpha=0.1)

                    xs, ys, es = zip(*[(yu.ens2a[ens]**2+ 0.001/10*ij, *yu.jackme(key2phy[(ens, j)])) for iens,ens in enumerate(enss)])
                    es=np.sqrt(np.array(es)**2 + np.array([key2syst[(ens, j)] for ens in enss])**2)
                    ax.errorbar(xs, ys, yerr=es, fmt=m, color=c, capsize=capsize_global, capthick=capthick_global, lw=2, ms=6)

        ax.axhline(ref, color="black", ls=":", lw=2, marker="")
        ax.set(ylabel=ylabel if ylabelQ else None, ylim=ylim)
        ax.set(ylabel=ylabel if ylabelQ else None, ylim=ylim, yticks=yticks)
        # ax.tick_params(direction="in", top=True, right=True)
        # ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=4))

        if which=='B20' and j=='jtot':
            pass
        else:
            ax.legend(loc="upper right", frameon=True, ncol=len(js),
                  fontsize=15 if which not in ['DeltaSigmaBy2'] else 14, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

    ax=axs[-1]
    if rightmostQ:
        ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(0, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
    else:
        ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(0, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007], xticklabels=[0,1,2,3,4,5,6,7])
    
def plot_A20_B20_J_v1(fig,axs,key2phy,which,ylabelQ=True,ce='final',key2syst=None,syst0onlyQ=True,rightmostQ=False):
    sty = {"jv1": ("r", "s")}

    if which == "A20":
        rows = [(['jv1'], r"$\langle x\rangle_{u-d}$", (0.12,0.22), [0.14,0.16,0.18,0.2], 1.0)]
        lab = dict(jv1=r"$\langle x\rangle_{u-d}$")
    elif which == "B20":
        rows = [(['jv1'], r"$B_{20}^{u-d}$", (0.1,0.3), [0.15,0.2,0.25], 1.0)]
        lab = dict(jv1=r"$B_{20}^{u-d}$")
    elif which == 'J':
        rows = [(['jv1'], r"$J_{u-d}$", (0.1,0.3), [0.15,0.2,0.25], 1.0)]
        lab = dict(jv1=r"$J_{u-d}$")
    elif which =='DeltaSigmaBy2':
        rows = [(['jv1'], r"$\frac{1}{2}\Delta\Sigma_{u-d}$", (0.1,0.3), [0.15,0.2,0.25], 1.0)]
        lab = dict(jv1=r"$L_{u-d}$")
    elif which =='L':
        rows = [(['jv1'], r"$L_{u-d}$", (0.1,0.3), [0.15,0.2,0.25], 1.0)]
        lab = dict(jv1=r"$L_{u-d}$")
    else:
        1/0

    for ax, (js, ylabel, ylim, yticks, ref) in zip(axs, rows):
        for ij,j in enumerate(js):
            c, m = sty[j]
            x = np.asarray(yum.lat_a2s_plt)
            y, e = map(np.asarray, yu.jackme(key2phy[("a=#_" + ce, j)]))

            ax.plot(x, y, "--", color=c, lw=2)
            ax.fill_between(x, y - e, y + e, color=c, alpha=0.2)

            xs, ys, es = zip(*[(yu.ens2a[ens]**2 + 0.001/10*ij, *yu.jackme(key2phy[(ens, j)])) for iens,ens in enumerate(enss)])
            ax.errorbar(xs, ys, yerr=es, fmt=m, color=c, label=lab[j], capsize=capsize_global, capthick=capthick_global, lw=2, ms=6)
            
            if key2syst is not None:
                x = np.asarray(yum.lat_a2s_plt)
                y, e = map(np.asarray, yu.jackme(key2phy[("a=#_" + ce, j)]))
                if syst0onlyQ:
                    ax.errorbar(0.001/5,y[0],e[0], color=c, fmt='*', markersize=16, capsize=capsize_global, capthick=capthick_global, mfc='white')
                    e = np.sqrt(e**2 + key2syst[("a=#_" + ce, j)]**2)
                    ax.errorbar(0.001/5,y[0],e[0], color=c, fmt='*', markersize=16, capsize=capsize_global, capthick=capthick_global, mfc='white')
                else:
                    e = np.sqrt(e**2 + key2syst[("a=#_" + ce, j)]**2)
                    ax.plot(x, y, "--", color=c, lw=2)
                    ax.fill_between(x, y - e, y + e, color=c, alpha=0.1)

                    xs, ys, es = zip(*[(yu.ens2a[ens]**2+ 0.001/10*ij, *yu.jackme(key2phy[(ens, j)])) for iens,ens in enumerate(enss)])
                    es=np.sqrt(np.array(es)**2 + np.array([key2syst[(ens, j)] for ens in enss])**2)
                    ax.errorbar(xs, ys, yerr=es, fmt=m, color=c, capsize=capsize_global, capthick=capthick_global, lw=2, ms=6)

        ax.axhline(ref, color="black", ls=":", lw=2, marker="")
        ax.set(ylabel=ylabel if ylabelQ else None, ylim=ylim, yticks=yticks)
        ax.tick_params(direction="in", top=True, right=True)
        
    ax=axs[-1]
    if rightmostQ:
        ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(0, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
    else:
        ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(0, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007], xticklabels=[0,1,2,3,4,5,6,7])
        
def run(key2phys,which,ratios,figsize):
    folders=['step1','case3','step2','case6']
    fig, axs = plt.subplots(1, len(folders), figsize=(6*len(folders),4), sharex='col', sharey='row',
                            gridspec_kw={"hspace": 0, 'wspace': 0.02})
    for i in range(len(folders)):
        plot_A20_B20_J_v1(fig,[axs[i]],key2phys[i],which,ylabelQ=(i==0),rightmostQ=(i==3))
    yu.finalizePlot(f'a2dep_{which}_compare_v1',tightQ=False)
    
    key2phy=key2phys[0]; key2syst=get_key2syst(key2phys); key2syst_2=get_key2syst_2(key2phys)
    
    fig, axs = plt.subplots(len(ratios), len(folders), figsize=(figsize[0]*len(folders),figsize[1]), sharex='col', sharey='row',
                            gridspec_kw={"height_ratios": ratios, "hspace": 0, 'wspace': 0.02})
    axs=np.atleast_2d(axs)
    for i in range(len(folders)):
        plot_A20_B20_J(fig,axs[:,i],key2phys[i],which,ylabelQ=(i==0),rightmostQ=(i==3),key2syst=key2syst if i==0 else None)
    yu.finalizePlot(f'a2dep_{which}_compare',tightQ=False) 
    
    fig, axs = plt.subplots(len(ratios), 1, figsize=figsize, sharex=True, sharey='row',
                            gridspec_kw={"height_ratios": ratios, "hspace": 0, 'wspace': 0.02})
    axs=np.atleast_1d(axs)
    plot_A20_B20_J(fig,axs,key2phy,which,rightmostQ=True)
    yu.finalizePlot(f'a2dep_{which}',tightQ=False)
    
    fig, axs = plt.subplots(len(ratios), 1, figsize=figsize, sharex=True, sharey='row',
                            gridspec_kw={"height_ratios": ratios, "hspace": 0, 'wspace': 0.02})
    axs=np.atleast_1d(axs)
    plot_A20_B20_J(fig,axs,key2phy,which,key2syst=key2syst,rightmostQ=True)
    yu.finalizePlot(f'a2dep_{which}_syst',tightQ=False)

    if which in ['A20','B20','J']:
        js_top = ['ju','jd','js','jc','jg','jtot']
        js_bot = ['jq','jv1','jv2','jv3','j+']
    elif which in ['L','DeltaSigmaBy2']:
        js_top = ['ju','jd','js','jc']
        js_bot = ['jq','jv1','jv2','jv3','j+']

    sub = {
        'ju':'u', 'jd':'d', 'js':'s', 'jc':'c', 'jg':'g', 'jtot':'N',
        'jv1':'u-d', 'j+':'u+d', 'jv2':'u+d-2s', 'jv3':'u+d+s-3c', 'jq':'u+d+s+c'
    }

    base = {'A20': r'\braket{x}', 'B20': r'B_{20}', 'J': r'J', 'L': r'L','DeltaSigmaBy2':r'\frac{1}{2}\Delta\Sigma'}[which]
    pos  = {'A20': '_', 'B20': '^', 'J': '_', 'L': '_', 'DeltaSigmaBy2': '_'}[which]

    rows = [yu.ens2label[e] for e in enss] + [r'$a=0$']

    def label(j):
        return rf'${base}{pos}{{{sub[j]}}}$'
    
    def f(j, ens=None):
        if ens is not None:
            return yu.jackme_un2str(key2phy[(ens,j)], precision=2)
        stat = yu.jackme_un2str(key2phy[('a=#_final',j)][:,0], precision=2)
        syst = tuple(ele[0] for ele in key2syst_2[('a=#_final',j)])
        return yu.me2mes(stat, syst)

    def block(js):
        df = pd.DataFrame(
            [[f(j,e) for j in js] for e in enss] + [[f(j) for j in js]],
            index=rows,
            columns=[label(j) for j in js],
        )
        tex = df.to_latex(
            escape=False,
            column_format='c' * (len(js) + 1),
            bold_rows=False,
        )
        tex = tex.replace(r'\toprule' + '\n', '')
        tex = tex.replace(r'\midrule', r'\hline')
        tex = tex.replace(r'\bottomrule' + '\n', '')
        tex = tex.replace('\n' + rows[-1] + ' &', '\n\\hline\n' + rows[-1] + ' &')
        return tex.strip()

    tex = block(js_top) + '\n\\vspace{2mm}\n' + block(js_bot)
    print(tex)
    print()

In [6]:
run(key2phy_A20s,'A20',[1.15, 1.0], (5.6, 7.4))
run(key2phy_B20s,'B20',[1.0, 1.0, 2.0], (5.6, 7.4))
run(key2phy_Js,'J',[1.15, 1.0], (5.6, 7.4))
run(key2phy_Ls,'L',[1], (5.6, 7.4/2))
run(key2phy_DeltaSigmaBy2s,'DeltaSigmaBy2', [1], (5.6, 7.4/2))

\begin{tabular}{ccccccc}
 & $\braket{x}_{u}$ & $\braket{x}_{d}$ & $\braket{x}_{s}$ & $\braket{x}_{c}$ & $\braket{x}_{g}$ & $\braket{x}_{N}$ \\
\hline
B64 & 0.357(24) & 0.199(21) & 0.0313(66) & 0.0034(53) & 0.382(27) & 0.973(59) \\
C80 & 0.359(20) & 0.189(17) & 0.0156(98) & -0.0121(81) & 0.430(39) & 0.981(67) \\
D96 & 0.377(23) & 0.214(21) & 0.048(17) & 0.023(15) & 0.303(56) & 0.96(10) \\
E112 & 0.355(33) & 0.191(30) & 0.072(24) & 0.047(21) & 0.373(69) & 1.04(13) \\
\hline
$a=0$ & 0.357(17)(11)(3) & 0.192(16)(2)(2) & 0.049(12)(5)(3) & 0.025(10)(6)(3) & 0.372(30)(8)(0)(16) & 0.995(60)(24)(1)(16) \\
\end{tabular}
\vspace{2mm}
\begin{tabular}{cccccc}
 & $\braket{x}_{u+d+s+c}$ & $\braket{x}_{u-d}$ & $\braket{x}_{u+d-2s}$ & $\braket{x}_{u+d+s-3c}$ & $\braket{x}_{u+d}$ \\
\hline
B64 & 0.590(47) & 0.158(15) & 0.493(40) & 0.577(41) & 0.556(42) \\
C80 & 0.551(48) & 0.170(11) & 0.517(29) & 0.599(30) & 0.548(35) \\
D96 & 0.661(71) & 0.164(11) & 0.495(24) & 0.571(26) & 0.591(42) \\
E112 & 0.66(10) 

In [7]:
def plot_bar(key2phy,which,key2syst=None):
    colors = dict(ju="red", jd="green", js="blue", jc="orange",
                  jq="purple", jg="cyan", jtot="gray")
    names = dict(ju=r"$u$", jd=r"$d$", js=r"$s$", jc=r"$c$",
                 jq=r"$q^s$", jg=r"$g$", jtot=r"$\mathrm{Total}$")

    setup = {
        "A20": (["ju", "jd", "js", "jc", "jq", "jg", "jtot"],
                r"$\langle x\rangle_{q,g}$", (0.0, 1.25),
                [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2], [1.0], 1.0, 2),
        "J": (["ju", "jd", "js", "jc", "jq", "jg", "jtot"],
              r"$J_{q,g}$", (0.0, 0.62),
              [0.0, 0.1, 0.2, 0.3, 0.4, 0.5], [0.5], 0.5, 0),
        "DeltaSigmaBy2": (["ju", "jd", "js", "jc", "jq"],
                          r"$\frac{1}{2}\Delta\Sigma_q$", (-0.32, 0.57),
                          [-0.25, 0.0, 0.25, 0.5], [0.0, 0.5], 0.5, -16),
        "L": (["ju", "jd", "js", "jc", "jq"],
              r"$L_{q}$", (-0.32, 0.57),
              [-0.25, 0.0, 0.25, 0.5], [0.0, 0.5], 0.5, -16),
    }

    js, ylabel, ylim, yticks, refs, norm, labelpad = setup[which]
    yr = ylim[1] - ylim[0]
    x = np.arange(len(js))

    fig, ax = plt.subplots(figsize=(7.2, 5.4))

    for i, j in enumerate(js):
        c = colors[j]

        m, e = yu.jackme(key2phy[("a=#_final", j)][:, 0])
        ax.bar(i, m, width=0.52, color=c, alpha=0.22, edgecolor=c, linewidth=1.3)
        ax.errorbar(i, m, yerr=e, fmt="none", color="black", capsize=6, lw=2, capthick=capthick_global)
        if key2syst is not None:
            m, e = yu.jackme(key2phy[("a=#_final", j)][:, 0])
            e = np.sqrt(e**2 + np.array([key2syst[("a=#_final", j)][0]])**2)[0]
            ax.errorbar(i, m, yerr=e, fmt="none", color="black", capsize=6, lw=2, capthick=capthick_global)

        jc = f"{j};conn"
        if ("a=#_final", jc) in key2phy:
            mc, _ = yu.jackme(key2phy[("a=#_final", jc)][:, 0])
            ax.bar(i, mc, width=0.32, color=c, alpha=0.85, edgecolor=c, linewidth=1.3)

        txt = rf"${100*m/norm:.1f}({100*e/norm:.1f})\%$"
        if key2syst is not None:
            m, e = yu.jackme(key2phy[("a=#_final", j)][:, 0])
            s = key2syst[("a=#_final", j)][0]
            txt = rf"${100*m/norm:.1f}({100*e/norm:.1f})({100*s/norm:.1f})\%$"
        
        ytxt = m + np.sign(m if m else 1) * (e + 0.035 * yr)
        ytxt = np.clip(ytxt, ylim[0] + 0.18 * yr, ylim[1] - 0.18 * yr)

        ax.text(i - 0.42, ytxt, txt, rotation=90,
                ha="center", va="center", fontsize=12, clip_on=True)

    for y in refs:
        ax.axhline(y, color="black", ls="--", lw=2, marker="")

    ax.set(
        ylabel=ylabel, ylim=ylim, yticks=yticks,
        xticks=x, xticklabels=[names[j] if not (which in ['DeltaSigmaBy2','L'] and j == 'jq') else names['jtot'] for j in js],
    )
    ax.set_xlim(-0.75, len(js) - 0.25)
    ax.yaxis.labelpad = labelpad
    ax.tick_params(direction="in", top=True, right=True)

    for s in ax.spines.values():
        s.set_linewidth(2)

    t={'A20':'avgx'}[which] if which in ['A20'] else which
    yu.finalizePlot(f'bar_{t}')
    return fig, ax

for which in ["A20", "J", "DeltaSigmaBy2", "L"]:
    plot_bar(globals()[f"key2phy_{which}s"][0],which,key2syst=get_key2syst(globals()[f"key2phy_{which}s"]) if which not in ['DeltaSigmaBy2'] else None)

In [8]:
js=['ju','jd','js','jc','jg','jtot']
whichs=['A20','J','DeltaSigmaBy2','L']

index=['u','d','s','c','g','Sum']
columns=[r'$\braket{x}$',r'$J$',r'$\frac{1}{2}\Delta\Sigma$',r'$L$']

def get(which,j):
    if j=='jtot' and which in ['DeltaSigmaBy2','L']:
        j='jq'
    key2phys=globals()[f'key2phy_{which}s']
    key2phy=key2phys[0]
    if which in ['DeltaSigmaBy2']:
        key2syst=None
    else:
        key2syst=get_key2syst_2(key2phys)
    key=('a=#_final',j)
    if key in key2phy:
        stat = yu.jackme_un2str(key2phy[key][:,0])
        if key2syst is not None:
            syst = tuple([ele[0] for ele in key2syst[('a=#_final',j)]])
            return yu.me2mes(stat, syst)
        return stat
    return ''

df = [[get(which,j) for which in whichs] for j in js]

df=pd.DataFrame(df,index=index,columns=columns)

tex=df.to_latex(
    escape=False,          # allow latex in labels
    column_format='cccccc',
    bold_rows=False,
)
tex = tex.replace(r'\toprule'+'\n',r'')
tex = tex.replace(r'\midrule', r'\hline')
tex = tex.replace(r'\bottomrule'+'\n',r'')
print(tex)

\begin{tabular}{cccccc}
 & $\braket{x}$ & $J$ & $\frac{1}{2}\Delta\Sigma$ & $L$ \\
\hline
u & 0.357(17)(11)(3) & 0.239(14)(20)(6) & 0.417(11) & -0.177(18)(20)(6) \\
d & 0.192(16)(2)(2) & 0.025(11)(10)(6) & -0.2033(89) & 0.228(15)(10)(6) \\
s & 0.049(12)(5)(3) & 0.0215(76)(123)(4) & -0.0187(51) & 0.0401(92)(123)(4) \\
c & 0.025(10)(6)(3) & 0.0121(66)(93)(22) & -0.0032(32) & 0.0153(74)(93)(22) \\
g & 0.372(30)(8)(0)(16) & 0.209(26)(58)(32)(0) &  &  \\
Sum & 0.995(60)(24)(1)(16) & 0.507(43)(63)(17)(0) & 0.190(22) & 0.108(37)(19)(15) \\
\end{tabular}



In [19]:
js=['ju','jd','js','jc','jg','jtot']
whichs=['A20','J','DeltaSigmaBy2','L']

index=['u','d','s','c','g','Sum']
columns=[r'$\braket{x}$',r'$J$',r'$\frac{1}{2}\Delta\Sigma$',r'$L$']

def get(which,j):
    if j=='jtot' and which in ['DeltaSigmaBy2','L']:
        j='jq'
    key2phys=globals()[f'key2phy_{which}s']
    key2phy=key2phys[0]
    if which in ['DeltaSigmaBy2']:
        key2syst=None
    else:
        key2syst=get_key2syst(key2phys)
    key=('a=#_final',j)
    if key in key2phy:
        stat = yu.jackme_un2str(key2phy[key][:,0])
        if key2syst is not None:
            syst = key2syst[('a=#_final',j)][0]
            return yu.me2mes(stat, syst)
        return stat
    return ''

df = [[get(which,j) for which in whichs] for j in js]

df=pd.DataFrame(df,index=index,columns=columns)

tex=df.to_latex(
    escape=False,          # allow latex in labels
    column_format='cccccc',
    bold_rows=False,
)
tex = tex.replace(r'\toprule'+'\n',r'')
tex = tex.replace(r'\midrule', r'\hline')
tex = tex.replace(r'\bottomrule'+'\n',r'')
print(tex)

\begin{tabular}{cccccc}
 & $\braket{x}$ & $J$ & $\frac{1}{2}\Delta\Sigma$ & $L$ \\
\hline
u & 0.357(17)(11) & 0.239(14)(21) & 0.417(11) & -0.177(18)(21) \\
d & 0.192(16)(3) & 0.025(11)(12) & -0.2033(89) & 0.228(15)(12) \\
s & 0.049(12)(6) & 0.0215(76)(123) & -0.0187(51) & 0.0401(92)(123) \\
c & 0.025(10)(7) & 0.0121(66)(95) & -0.0032(32) & 0.0153(73)(95) \\
g & 0.372(30)(18) & 0.209(26)(66) &  &  \\
Sum & 0.995(60)(29) & 0.507(43)(65) & 0.190(22) & 0.108(37)(24) \\
\end{tabular}



# compare

In [20]:
pdflabels=['HERAPDF2.0','ABMP16','CT18','MSHT20','NNPDF4.0','PDF4LHC21','JAM22','CJ22']
pdfsets=['HERAPDF20_NNLO_EIG','ABMP16_4_nnlo','CT18NNLO','MSHT20nnlo_as118','NNPDF40_nnlo_as_01180','PDF4LHC21_40_pdfas','JAM22-PDF_proton_nlo','CJ22']
label2j2me_A20=yu.load_pkl_reg('label2j2me_avgx',pathlabel='lhapdf')
label2j2me_A20={label:{j[:-2] if j.endswith(';+') else j:me for j,me in label2j2me_A20[label].items()} for label in label2j2me_A20.keys()}

pdflabels_pol=['DSSV14','JAM22','NNPDFpol2.0']
pdfsets_pol=['DSSV14pol','JAM22-PPDF_proton_nlo','NNPDFpol20_nnlo_as_01180_mhou']
label2j2me_gA=yu.load_pkl_reg('label2j2me_gA_xmin=1e-3',pathlabel='lhapdf')
label2j2me_gA={label:{j[:-2] if j.endswith(';+') else j:me for j,me in label2j2me_gA[label].items()} for label in label2j2me_gA.keys()}

label2j2me_DeltaSigmaBy2={label:{j:(m/2,e/2) for j,(m,e) in label2j2me_gA[label].items()} for label in label2j2me_gA.keys()}

In [21]:
def plot_compare(which):
    key2phy = globals()[f"key2phy_{which}s"][0]
    get_src = getattr(yum, f"get_{which}_from_src")
    
    if which in ['DeltaSigmaBy2']:
        key2syst=None
    else:
        key2syst = get_key2syst(globals()[f"key2phy_{which}s"])
    

    cfg = {
        "A20": dict(
            js=["ju", "jd", "js", "jc", "jg"],
            xlabel=[r"$\langle x\rangle_u$", r"$\langle x\rangle_d$", r"$\langle x\rangle_s$",
                    r"$\langle x\rangle_c$", r"$\langle x\rangle_g$"],
            phen=(["ABMP16", "CT18", "MSHT20", "NNPDF4.0", "JAM22", "CJ22"],
                  pdflabels, pdfsets, label2j2me_A20),
            srcs=[r"$\chi$QCD18", "MIT24", "ETM20"],
            xlim=[(0.22, 0.52), (0.06, 0.36), (-0.10, 0.20), (-0.12, 0.18), (0.10, 0.60)],
            xticks=[[0.30, 0.45], [0.15, 0.30], [0.0, 0.1], [0.0, 0.1], [0.25, 0.45]],
            figsize=(13.5, 3.5),
        ),
        "J": dict(
            js=["ju", "jd", "js", "jc", "jg"],
            xlabel=[r"$J_u$", r"$J_d$", r"$J_s$", r"$J_c$", r"$J_g$"],
            phen=None, srcs=["MIT24", "ETM20"],
            xlim=[(0.12, 0.32), (-0.04, 0.16), (-0.08, 0.12), (-0.09, 0.09), (0.10, 0.30)],
            xticks=[[0.18, 0.28], [0.0, 0.1], [0.0, 0.08], [0.0, 0.07], [0.15, 0.25]],
            figsize=(13.5, 1.1),
        ),
        
        "DeltaSigmaBy2": dict(
            js=["ju", "jd", "js", "jc"],
            xlabel=[r"$\frac{1}{2}\Delta\Sigma_u$", r"$\frac{1}{2}\Delta\Sigma_d$",
                    r"$\frac{1}{2}\Delta\Sigma_s$", r"$\frac{1}{2}\Delta\Sigma_c$"],
            phen=(["DSSV14", "JAM22", "NNPDFpol2.0"],
                  pdflabels_pol, pdfsets_pol, label2j2me_DeltaSigmaBy2),
            srcs=[r"$\chi$QCD18", "PNDME25", "Mainz26", "ETM20"],
            xlim=[(0.32, 0.52), (-0.31, -0.11), (-0.12, 0.10), (-0.09, 0.09)],
            xticks=[[0.35, 0.45], [-0.25, -0.15], [-0.08, 0.0], [-0.06, 0.0, 0.06]],
            figsize=(11.5, 2.8),
        ),
        "L": dict(
            js=["ju", "jd", "js", "jc"],
            xlabel=[r"$L_u$", r"$L_d$", r"$L_s$", r"$L_c$"],
            phen=None, srcs=["ETM20"],
            xlim=[(-0.30, -0.10), (0.16, 0.36), (-0.04, 0.16), (-0.09, 0.09)],
            xticks=[[-0.25, -0.15], [0.2, 0.3], [0.0, 0.1], [0.0, 0.07]],
            figsize=(11.5, 0.8),
        ),
    }[which]

    def missing(v):
        return v is None or (isinstance(v, tuple) and any(x is None for x in v))

    def split_err(e):
        if isinstance(e, tuple):
            estat, esys = e
            return estat, np.sqrt(estat**2 + esys**2)
        return None, e

    phen_rows = []
    if cfg["phen"] is not None:
        order, pdflabs, pdfsets_, label2j2me = cfg["phen"]
        lab2set = dict(zip(pdflabs, pdfsets_))
        phen_rows = [(lab, lab2set[lab]) for lab in order if lab in lab2set]

    ylabels = [lab for lab, _ in phen_rows] + cfg["srcs"] + ["This Work"]
    y = np.arange(len(ylabels))
    ymap = dict(zip(ylabels, y))

    fig, axs = plt.subplots(1, len(cfg["js"]), figsize=cfg["figsize"], sharey=True, gridspec_kw={"wspace": 0.05})
    axs = np.atleast_1d(axs)

    for ax, j, xl, xlim, xticks in zip(axs, cfg["js"], cfg["xlabel"], cfg["xlim"], cfg["xticks"]):
        m, e = yu.jackme(key2phy[("a=#_final", j)][:, 0])
        
        ax.errorbar(m, ymap["This Work"], xerr=e, fmt="s", color="red", capthick=capthick_global)
        if key2syst is not None:
            e = np.sqrt(e**2 + key2syst[("a=#_final", j)][0]**2)
            ax.errorbar(m, ymap["This Work"], xerr=e, fmt="s", color="red", capthick=capthick_global)
        
        ax.axvspan(m - e, m + e, color="red", alpha=0.20)
        # ax.axvline(m, color="red", alpha=0.45)
    
        if cfg["phen"] is not None:
            label2j2me = cfg["phen"][3]
            for lab, label in phen_rows:
                if which == "DeltaSigmaBy2" and j == "jc" and lab in ["DSSV14", "JAM22"]:
                    continue
                val = label2j2me.get(label, {}).get(j)
                if missing(val):
                    continue
                mp, ep = val
                if missing((mp, ep)):
                    continue
                _, ep = split_err(ep)
                ax.errorbar(mp, ymap[lab], xerr=ep, fmt="^", color="black", capthick=capthick_global)

        for src in cfg["srcs"]:
            val = get_src(src, j)
            if missing(val):
                continue
            ms, es = val
            if missing((ms, es)):
                continue
            estat, etot = split_err(es)

            if src == "ETM20":
                col, mk, mfc = "green", "o", "white"
            elif src == "MIT24":
                col, mk, mfc = "blue", "d", "white"
            else:
                col, mk, mfc = "blue", "d", "blue"

            ax.errorbar(ms, ymap[src], xerr=etot, fmt=mk, color=col, mfc=mfc, mec=col, capthick=capthick_global)
            if estat is not None:
                ax.errorbar(ms, ymap[src], xerr=estat, fmt="none", color=col, capthick=capthick_global)

        ax.set(xlabel=xl, xlim=xlim, xticks=xticks)
        ax.set_ylim(len(ylabels) - 0.25, -0.75)
        ax.tick_params(direction="in", top=True, right=True, labelsize=16)
        for s in ax.spines.values():
            s.set_linewidth(2)

    axs[0].set_yticks(y)
    axs[0].set_yticklabels(ylabels, fontsize=18)
    for ax in axs[1:]:
        ax.tick_params(labelleft=False)

    t={'A20':'avgx'}[which] if which in ['A20'] else which
    yu.finalizePlot(f'compare_{t}',tightQ=False)
    return fig, axs


for which in ["A20", "J", "DeltaSigmaBy2", "L"]:
    plot_compare(which)